In [ ]:
from ipywidgets import IntSlider, HBox, Layout, VBox, HTML
from IPython.display import display


def sign_magnitude(x, bits):
    magnitude = abs(x)

    if magnitude > 2**(bits - 1) - 1:
        return None

    sign = '1' if x < 0 else '0'
    magnitude_bits = format(magnitude, f'0{bits - 1}b')

    return sign + magnitude_bits


def ones_complement(x, bits):
    if x >= 0:
        return format(x, f'0{bits}b')

    positive_bits = format(abs(x), f'0{bits}b')
    complement = ''.join('1' if bit == '0' else '0' for bit in positive_bits)

    return complement


def twos_complement(x, bits):
    if x >= 0:
        return format(x, f'0{bits}b')

    return format((1 << bits) + x, f'0{bits}b')


summary_html = HTML()

representation_html = HTML()


slider_layout = Layout(width='300px')

style_opts = {'description_width': '90px'}


bits_slider = IntSlider(
    min=4,
    max=12,
    step=1,
    value=5,
    description='Word length:',
    style=style_opts,
    layout=slider_layout,
    continuous_update=True
)


initial_limit = 2**(bits_slider.value - 1) - 1


x_slider = IntSlider(
    min=-initial_limit,
    max=initial_limit,
    step=1,
    value=-5,
    description='Integer x:',
    style=style_opts,
    layout=slider_layout,
    continuous_update=True
)


def update_display(*args):
    x = x_slider.value
    bits = bits_slider.value

    common_limit = 2**(bits - 1) - 1

    sign_mag = sign_magnitude(x, bits)
    ones = ones_complement(x, bits)
    twos = twos_complement(x, bits)

    sign_mag_display = sign_mag[0] + " " + sign_mag[1:]
    ones_display = ones[0] + " " + ones[1:]
    twos_display = twos[0] + " " + twos[1:]

    summary_html.value = f"""
    <div style="
        font-family: monospace;
        font-size: 15px;
        line-height: 2.0;
        white-space: nowrap;
    ">
    <b>Decimal value:</b> {x}<br>
    <b>Word length:</b> {bits} bits
    </div>
    """

    representation_html.value = f"""
    <div style="
        font-family: monospace;
        font-size: 15px;
        line-height: 2.0;
        margin-top: 18px;
    ">

    <table style="border-collapse: collapse;">

    <tr>
    <td style="padding: 8px 25px;">
    <b>Representation</b>
    </td>

    <td style="padding: 8px 25px;">
    <b>Binary word</b>
    </td>
    </tr>

    <tr>
    <td style="padding: 8px 25px;">
    Sign magnitude
    </td>

    <td style="padding: 8px 25px; font-size: 18px;">
    {sign_mag_display}
    </td>
    </tr>

    <tr>
    <td style="padding: 8px 25px;">
    One's complement
    </td>

    <td style="padding: 8px 25px; font-size: 18px;">
    {ones_display}
    </td>
    </tr>

    <tr>
    <td style="padding: 8px 25px;">
    Two's complement
    </td>

    <td style="padding: 8px 25px; font-size: 18px;">
    {twos_display}
    </td>
    </tr>

    </table>

    <br>

    <b>Common representable range:</b>
    [{-common_limit}, {common_limit}]

    </div>
    """


def update_integer_range(change):
    bits = change['new']

    limit = 2**(bits - 1) - 1

    current_value = x_slider.value

    x_slider.min = -limit
    x_slider.max = limit

    if current_value < -limit:
        x_slider.value = -limit

    elif current_value > limit:
        x_slider.value = limit

    update_display()


bits_slider.observe(update_integer_range, names='value')

x_slider.observe(update_display, names='value')


theory_html = HTML("""
<div style="
    font-family: monospace;
    font-size: 13px;
    line-height: 1.55;
    margin-bottom: 12px;
">

<b>Sign magnitude:</b> One bit represents the sign and the remaining bits represent the magnitude.<br><br>

<b>One's complement:</b> A negative number is obtained by inverting every bit of its positive representation.<br><br>

<b>Two's complement:</b> A negative number is obtained by adding one to the one's-complement representation.<br><br>

<b>Integer x:</b> Selects the decimal integer whose three binary representations are displayed.
Its allowable range is automatically updated whenever the word length changes.<br><br>

<b>Word length:</b> Determines the total number of bits used in each representation.
Increasing the word length increases the representable numerical range exponentially.
For example, the common range is [-63, 63] for 7 bits and [-2047, 2047] for 12 bits.<br><br>

The displayed range is restricted to the common range supported by all three representations,
so that the same integer can always be compared directly.

</div>
""")


summary_html.layout = Layout(
    width='165px',
    min_width='165px'
)


controls = VBox(
    [x_slider, bits_slider],
    layout=Layout(
        width='305px',
        min_width='305px',
        overflow='visible',
        justify_content='flex-start'
    )
)


top_row = HBox(
    [summary_html, controls],
    layout=Layout(
        width='500px',
        align_items='flex-start',
        justify_content='flex-start',
        overflow='visible'
    )
)


main_content = VBox(
    [top_row, representation_html],
    layout=Layout(
        width='650px',
        align_items='flex-start',
        overflow='visible'
    )
)


update_display()


display(
    VBox(
        [theory_html, main_content],
        layout=Layout(
            width='100%',
            align_items='flex-start',
            overflow='visible'
        )
    )
)